# Train a snail chess network 🐌

Trains an NNUE network for one [Immortal Snail Chess](https://chess.stbinf.com/) variant and tests it.

1. **Runtime → Change runtime type → a GPU** (T4 works; L4/A100 are faster).
2. Set the options in the first cell, then **Runtime → Run all**.

Training data comes from the **Snail training data** workflow in
[sirsigmund79/variant-nnue-tools](https://github.com/sirsigmund79/variant-nnue-tools/actions/workflows/snail-data.yml)
(run it first for the variant and generation you want). Everything you keep (checkpoints, the network, match
results) goes to Google Drive under `MyDrive/snail-bots/`, so if Colab disconnects just **Run all** again:
training picks up from the last checkpoint.

In [ ]:
#@title Settings
VARIANT = "snail-3-d-u"  #@param ["snail-3-d-u", "snail-3-d-w", "snail-3-s-u", "snail-3-s-w", "snail-2-d-u", "snail-2-d-w", "snail-2-s-u", "snail-2-s-w", "snail-4-d-u", "snail-4-d-w", "snail-4-s-u", "snail-4-s-w"]
GENERATION = 0  #@param {type:"integer"}
#@markdown Leave empty to use the newest data release for this variant and generation, or give an exact tag.
DATA_RELEASE = ""  #@param {type:"string"}
#@markdown Training length: an epoch is EPOCH_SIZE positions (the data is reused cyclically).
EPOCHS = 100  #@param {type:"integer"}
EPOCH_SIZE = 10000000  #@param {type:"integer"}
BATCH_SIZE = 16384  #@param {type:"integer"}
#@markdown Optional: start from an earlier network's .pt on Drive (fine-tuning another variant or generation).
INIT_FROM_PT = ""  #@param {type:"string"}
#@markdown The match: games against OPPONENT_NET (a .nnue path on Drive), or the hand-written evaluation if empty.
MATCH_GAMES = 200  #@param {type:"integer"}
OPPONENT_NET = ""  #@param {type:"string"}

In [ ]:
#@title Google Drive and GPU
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = f"/content/drive/MyDrive/snail-bots/{VARIANT}-gen{GENERATION}"
os.makedirs(WORK, exist_ok=True)
print("Saving to", WORK)
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"

In [ ]:
#@title Install the trainer
%cd /content
!test -d variant-nnue-pytorch || git clone -q -b snail https://github.com/sirsigmund79/variant-nnue-pytorch
%cd /content/variant-nnue-pytorch
!git pull -q
# Colab's own PyTorch (CUDA 12) is used as is; the trainer needs Lightning 1.9.x, which needs an older setuptools.
!pip install -q "pytorch-lightning==1.9.5" "setuptools<70" chess
# CuPy matching Colab's CUDA: use the one Colab ships if there is one, else install it.
!python -c "import cupy" 2>/dev/null || pip install -q cupy-cuda12x
!sh compile_data_loader.bat > /dev/null 2>&1; ls *.so

In [ ]:
#@title Download the training data
import json, urllib.request
releases = json.load(urllib.request.urlopen("https://api.github.com/repos/sirsigmund79/variant-nnue-tools/releases?per_page=100"))
prefix = f"data-{VARIANT}-gen{GENERATION}-"
matches = [r for r in releases if (r["tag_name"] == DATA_RELEASE if DATA_RELEASE else r["tag_name"].startswith(prefix))]
if not matches:
    raise RuntimeError(f"No data release for {VARIANT} generation {GENERATION} yet. Run the 'Snail training data' workflow: "
                     "https://github.com/sirsigmund79/variant-nnue-tools/actions/workflows/snail-data.yml")
release = matches[0]  # newest first
print("Using", release["tag_name"], "-", release["body"].splitlines()[0])
os.makedirs("/content/data", exist_ok=True)
for asset in release["assets"]:
    path = f"/content/data/{asset['name']}"
    if not os.path.exists(path) or os.path.getsize(path) != asset["size"]:
        print("Downloading", asset["name"], f"({asset['size'] / 1e6:.0f} MB)")
        urllib.request.urlretrieve(asset["browser_download_url"], path)
TRAIN = f"/content/data/{VARIANT}-gen{GENERATION}.bin"
VAL = f"/content/data/{VARIANT}-gen{GENERATION}-val.bin"
print(TRAIN, os.path.getsize(TRAIN) // 72, "positions")
print(VAL, os.path.getsize(VAL) // 72, "positions")

In [ ]:
#@title Train (runs in the background; re-run this cell after a reconnect to see progress again)
# Training runs as its own process and logs to Drive, so a dropped browser connection doesn't
# stop it or lose its output. If the whole runtime was reset, run all cells again: training
# resumes from the last checkpoint on Drive.
import glob, re, subprocess, time
%cd /content/variant-nnue-pytorch
LOG = f"{WORK}/train.log"
def last_checkpoint():
    ckpts = glob.glob(f"{WORK}/lightning_logs/version_*/checkpoints/last.ckpt")
    return max(ckpts, key=os.path.getmtime) if ckpts else None
def training_running():
    return subprocess.run(["pgrep", "-f", "python train.py"], capture_output=True).returncode == 0
if not training_running():
    resume = last_checkpoint()
    extra = f"--resume_from_checkpoint {resume}" if resume else (f"--resume-from-model {INIT_FROM_PT}" if INIT_FROM_PT else "")
    print("Resuming from", resume) if resume else print("Starting fresh" + (f" from {INIT_FROM_PT}" if INIT_FROM_PT else ""))
    cmd = (f"nohup python train.py {TRAIN} {VAL} --gpus 1 --threads 4 --num-workers 4 --batch-size {BATCH_SIZE} "
           f"--random-fen-skipping 3 --features=HalfKAv2^ --max_epochs {EPOCHS} --epoch-size {EPOCH_SIZE} "
           f"--default_root_dir {WORK} --last-checkpoint-only {extra} >> {LOG} 2>&1 &")
    subprocess.run(cmd, shell=True)
    time.sleep(5)
else:
    print("Training is already running; showing its progress")
# Show one line per minute: the latest epoch and loss from the log.
last = ""
while training_running():
    time.sleep(60)
    tail = open(LOG, errors="ignore").read()[-20000:]
    found = re.findall(r"Epoch (\d+):\s+(\d+)%.*?loss=([0-9.e-]+)", tail)
    if found:
        epoch, pct, loss = found[-1]
        line = f"{time.strftime('%H:%M')}  epoch {epoch}/{EPOCHS}  {pct}%  loss {loss}"
        if line != last:
            print(line, flush=True)
            last = line
print(open(LOG, errors="ignore").read()[-1500:])
print("Training finished" if last_checkpoint() else "Training stopped without a checkpoint: see the log above")

In [ ]:
#@title Export the network
# The engine only loads a network whose file name starts with the variant name.
NET = f"{WORK}/{VARIANT}-gen{GENERATION}.nnue"
ckpt = last_checkpoint()
!python serialize.py {ckpt} {NET}
!python serialize.py {ckpt} {WORK}/{VARIANT}-gen{GENERATION}.pt
print("Network:", NET, f"({os.path.getsize(NET) / 1e6:.0f} MB)")

In [ ]:
#@title Build the engine and check its rules
%cd /content
!test -d Fairy-Stockfish || git clone -q -b snail https://github.com/sirsigmund79/Fairy-Stockfish
%cd /content/Fairy-Stockfish
!git pull -q
!cd src && make -j$(nproc) ARCH=x86-64-avx2 build > /dev/null 2>&1 && ls -la stockfish
!python setup.py build_ext --inplace > /dev/null 2>&1
!python tests/snail/compare.py tests/snail/positions.jsonl.gz | tail -1

In [ ]:
#@title Match: the new network against the opponent
opponent = f"--b-opt EvalFile={OPPONENT_NET}" if OPPONENT_NET else '--b-opt "Use NNUE=false"'
print("A = new network, B =", OPPONENT_NET or "hand-written evaluation")
!python tests/snail/match.py --variant {VARIANT} --games {MATCH_GAMES} --movetime 50 \
    --a src/stockfish --a-opt EvalFile={NET} --b src/stockfish {opponent} | tee {WORK}/match.txt

In [ ]:
#@title Download the network
from google.colab import files
files.download(NET)

## Next

- **Keep it** only if the match's whole Elo interval is above zero (more games narrow it).
- **Next generation:** upload the `.nnue` to a release in your `variant-nnue-tools` fork (Releases → Draft a new release → attach the file),
  copy its download link, and run the **Snail training data** workflow with `generation` 1 and `net_url` set to that link.
  Then set `GENERATION = 1`, `OPPONENT_NET` to this network on Drive, and run this notebook again.
- **Other variants:** run the workflow for that variant, set `VARIANT`, and `INIT_FROM_PT` to the standard network's `.pt` to start from it.